# 02 — Data Cleaning & Quality Assessment

## Objective

The goal of this stage is to assess and improve the quality of the dataset before feature engineering and model development.

Rather than applying transformations automatically, each cleaning decision will be based on the findings from the initial exploration and the actual characteristics of the data.

## Key Questions

- Are the duplicated rows valid observations or true duplicates?
- Are the feature values within valid and expected ranges?
- Are there unexpected or invalid categorical values?
- Are the binary features correctly represented?
- Do the ordinal features follow the expected ordering?
- Are there any hidden data-quality issues that were not identified during the initial exploration?
- Which data-cleaning decisions are justified by the evidence?

In [1]:
import numpy as np
import pandas as pd

## Load Data

In [2]:
df = pd.read_csv("../data/raw/diabetes_binary_health_indicators_BRFSS2015.csv")

In [3]:
df.head().T

,0,1,2,3,4
Diabetes_binary,0.0,0.0,0.0,0.0,0.0
HighBP,1.0,0.0,1.0,1.0,1.0
HighChol,1.0,0.0,1.0,0.0,1.0
CholCheck,1.0,0.0,1.0,1.0,1.0
BMI,40.0,25.0,28.0,27.0,24.0
Smoker,1.0,1.0,0.0,0.0,0.0
Stroke,0.0,0.0,0.0,0.0,0.0
HeartDiseaseorAttack,0.0,0.0,0.0,0.0,0.0
PhysActivity,0.0,1.0,0.0,1.0,1.0
Fruits,0.0,0.0,1.0,1.0,1.0


## 1. Duplicate Investigation

In this section, we investigate duplicate records and examine whether repeated feature combinations are associated with different target values.

In [4]:
df.duplicated().sum()

np.int64(24206)

In [5]:
print(f"duplicates percentage = {(df.duplicated().sum())/df.shape[0]}")

duplicates percentage = 0.09541942604856513


### 1.1 Target Conflicts

We investigate whether identical feature combinations can have different target values, which may indicate conflicting observations in the dataset.

In [6]:
feature_columns = df.columns.drop("Diabetes_binary")

target_variations = (
    df.groupby(list(feature_columns))["Diabetes_binary"]
      .nunique()
)

target_conflicts = target_variations[target_variations > 1]

print(f"Feature combinations with different targets: {len(target_conflicts)}")

Feature combinations with different targets: 1566


In [7]:
pd.DataFrame(target_conflicts).T

HighBP                0.0                                               ...  \
HighChol              0.0                                               ...   
CholCheck             1.0                                               ...   
BMI                  18.0 19.0      20.0                21.0            ...   
Smoker                0.0  0.0       0.0            1.0  0.0            ...   
Stroke                0.0  0.0       0.0            0.0  0.0            ...   
HeartDiseaseorAttack  0.0  0.0       0.0            0.0  0.0            ...   
PhysActivity          1.0  0.0  1.0  1.0            1.0  1.0            ...   
Fruits                1.0  1.0  1.0  0.0  1.0       1.0  0.0       1.0  ...   
Veggies               1.0  1.0  1.0  1.0  1.0       1.0  1.0       1.0  ...   
HvyAlcoholConsump     0.0  0.0  1.0  0.0  0.0       0.0  0.0       0.0  ...   
AnyHealthcare         1.0  1.0  1.0  1.0  1.0       1.0  1.0       1.0  ...   
NoDocbcCost           0.0  0.0  0.0  0.0  0.0       0.0  0.0       0.0  ...   
GenHlth               2.0  3.0  1.0  2.0  2.0       3.0  2.0  3.0  2.0  ...   
MentHlth              5.0  0.0  2.0  0.0  0.0       0.0  0.0  0.0  0.0  ...   
PhysHlth              0.0  0.0  0.0  0.0  0.0       0.0  1.0  0.0  0.0  ...   
DiffWalk              0.0  0.0  0.0  0.0  0.0       0.0  0.0  0.0  0.0  ...   
Sex                   0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  ...   
Age                  5.0  4.0  4.0  7.0  3.0  13.0 7.0  7.0  2.0  12.0  ...   
Education             6.0  6.0  6.0  6.0  6.0  6.0  6.0  6.0  6.0  6.0  ...   
Income                8.0  8.0  8.0  8.0  8.0  7.0  8.0  8.0  7.0  8.0  ...   
Diabetes_binary         2    2    2    2    2    2    2    2    2    2  ...   

HighBP                1.0                                               
HighChol              1.0                                               
CholCheck             1.0                                               
BMI                  37.0                          38.0 39.0 40.0 42.0  
Smoker                0.0                           1.0  0.0  0.0  0.0  
Stroke                0.0                           0.0  0.0  0.0  0.0  
HeartDiseaseorAttack  0.0                           0.0  0.0  0.0  0.0  
PhysActivity          1.0                           1.0  1.0  0.0  1.0  
Fruits                0.0  1.0                      1.0  1.0  0.0  1.0  
Veggies               1.0  1.0                      1.0  1.0  1.0  1.0  
HvyAlcoholConsump     0.0  0.0                      0.0  0.0  0.0  0.0  
AnyHealthcare         1.0  1.0                      1.0  1.0  1.0  1.0  
NoDocbcCost           0.0  0.0                      0.0  0.0  0.0  0.0  
GenHlth               3.0  2.0       3.0            3.0  3.0  2.0  3.0  
MentHlth              0.0  0.0       0.0            0.0  0.0  0.0  0.0  
PhysHlth              0.0  0.0       0.0            0.0  0.0  0.0  0.0  
DiffWalk              0.0  0.0       0.0            0.0  0.0  0.0  0.0  
Sex                   1.0  0.0       0.0  1.0       1.0  1.0  1.0  0.0  
Age                  6.0  9.0  10.0 8.0  7.0  9.0  10.0 7.0  10.0 9.0   
Education             6.0  5.0  5.0  6.0  6.0  5.0  5.0  4.0  5.0  4.0  
Income                8.0  6.0  7.0  8.0  8.0  7.0  8.0  8.0  8.0  6.0  
Diabetes_binary         2    2    2    2    2    2    2    2    2    2  

[1 rows x 1566 columns]

In [8]:
conflicting_combinations = target_variations[target_variations > 1]

print(conflicting_combinations.sum())

3132


In [9]:
conflicting_features = target_variations[target_variations > 1].index

conflicting_rows = df.set_index(list(feature_columns)).loc[conflicting_features]

print(f"Rows involved in target conflicts: {len(conflicting_rows)}")

Rows involved in target conflicts: 5218


In [10]:
duplicate_mask = df.duplicated(keep=False)

print(f"Rows belonging to exact duplicate groups: {duplicate_mask.sum()}")

Rows belonging to exact duplicate groups: 35575


## 2. Feature Value Validation

### 2.1 Binary Features

We verify that binary features contain only their expected values (`0` and `1`).

In [11]:
cols = df.columns
for col in cols [:10]:
    if df[col].nunique() == 2 :
        print(col)
        print(df[col].nunique(),"==========>",df[col].unique())
        print("________________________________")

Diabetes_binary
2 ==========> [0. 1.]
________________________________
HighBP
2 ==========> [1. 0.]
________________________________
HighChol
2 ==========> [1. 0.]
________________________________
CholCheck
2 ==========> [1. 0.]
________________________________
Smoker
2 ==========> [1. 0.]
________________________________
Stroke
2 ==========> [0. 1.]
________________________________
HeartDiseaseorAttack
2 ==========> [0. 1.]
________________________________
PhysActivity
2 ==========> [0. 1.]
________________________________
Fruits
2 ==========> [0. 1.]
________________________________


In [12]:
cols = df.columns
for col in cols [10:]:
    if df[col].nunique() == 2 :
        print(col)
        print(df[col].nunique(),"==========>",df[col].unique())
        print("________________________________")

Veggies
2 ==========> [1. 0.]
________________________________
HvyAlcoholConsump
2 ==========> [0. 1.]
________________________________
AnyHealthcare
2 ==========> [1. 0.]
________________________________
NoDocbcCost
2 ==========> [0. 1.]
________________________________
DiffWalk
2 ==========> [1. 0.]
________________________________
Sex
2 ==========> [0. 1.]
________________________________


### 2.2 Categorical / Ordinal Features

We inspect the unique values of categorical and ordinal features and verify that they fall within their expected ranges.

In [13]:
cols = df.columns
for col in cols :
    if df[col].nunique() > 2 and df[col].nunique() <15 :
        print(col)
        print(df[col].nunique(),"==========>",df[col].unique())
        print("________________________________")

GenHlth
5 ==========> [5. 3. 2. 4. 1.]
________________________________
Age
13 ==========> [ 9.  7. 11. 10.  8. 13.  4.  6.  2. 12.  5.  1.  3.]
________________________________
Education
6 ==========> [4. 6. 3. 5. 2. 1.]
________________________________
Income
8 ==========> [3. 1. 8. 6. 4. 7. 2. 5.]
________________________________


### 2.3 Continuous Features

We examine the descriptive statistics of continuous features, including their minimum, maximum, mean, median, quartiles, and standard deviation.

In [14]:
cols = df.columns
for col in cols :
    if df[col].nunique() > 15 :
        print(col)
        print(df[col].nunique(),"==========>",df[col].unique())
        print("________________________________")

BMI
84 ==========> [40. 25. 28. 27. 24. 30. 34. 26. 33. 21. 23. 22. 38. 32. 37. 31. 29. 20.
 35. 45. 39. 19. 47. 18. 36. 43. 55. 49. 42. 17. 16. 41. 44. 50. 59. 48.
 52. 46. 54. 57. 53. 14. 15. 51. 58. 63. 61. 56. 74. 62. 64. 66. 73. 85.
 60. 67. 65. 70. 82. 79. 92. 68. 72. 88. 96. 13. 81. 71. 75. 12. 77. 69.
 76. 87. 89. 84. 95. 98. 91. 86. 83. 80. 90. 78.]
________________________________
MentHlth
31 ==========> [18.  0. 30.  3.  5. 15. 10.  6. 20.  2. 25.  1.  4.  7.  8. 21. 14. 26.
 29. 16. 28. 11. 12. 24. 17. 13. 27. 19. 22.  9. 23.]
________________________________
PhysHlth
31 ==========> [15.  0. 30.  2. 14. 28.  7. 20.  3. 10.  1.  5. 17.  4. 19.  6. 12. 25.
 27. 21. 22.  8. 29. 24.  9. 16. 18. 23. 13. 26. 11.]
________________________________


## 3. Outlier Investigation

We investigate statistically unusual values and determine whether they represent actual data quality issues or valid extreme observations.

In [15]:
df.BMI.describe()

count    253680.000000
mean         28.382364
std           6.608694
min          12.000000
25%          24.000000
50%          27.000000
75%          31.000000
max          98.000000
Name: BMI, dtype: float64

In [16]:
Q1 = df["BMI"].quantile(0.25)
Q3 = df["BMI"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["BMI"] < lower) | (df["BMI"] > upper)]

print(len(outliers))

9847


In [17]:
outliers["BMI"].describe()

count    9847.000000
mean       48.510815
std         9.656683
min        12.000000
25%        43.000000
50%        45.000000
75%        50.000000
max        98.000000
Name: BMI, dtype: float64

### 3.1 MentHlth and PhysHlth

We investigate the statistical outliers in `MentHlth` and `PhysHlth` while considering their valid range of `0–30`.

In [18]:
Q1 = df["MentHlth"].quantile(0.25)
Q3 = df["MentHlth"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["MentHlth"] < lower) | (df["MentHlth"] > upper)]

print(len(outliers))

36208


In [19]:
outliers["MentHlth"].describe()

count    36208.000000
mean        19.079154
std          8.978813
min          6.000000
25%         10.000000
50%         15.000000
75%         30.000000
max         30.000000
Name: MentHlth, dtype: float64

In [20]:
Q1 = df["PhysHlth"].quantile(0.25)
Q3 = df["PhysHlth"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df["PhysHlth"] < lower) | (df["PhysHlth"] > upper)]

print(len(outliers))

40949


In [21]:
outliers["PhysHlth"].describe()

count    40949.000000
mean        22.313268
std          8.235429
min          8.000000
25%         15.000000
50%         25.000000
75%         30.000000
max         30.000000
Name: PhysHlth, dtype: float64

## 4. Missing Value Analysis

We check all features for missing values and determine whether any missing-value treatment is required.

In [22]:
df.isna().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

## 5. Cleaning Decisions

Based on the data quality investigation, the following decisions were made:

### 5.1 Exact Duplicates
Exact duplicate rows will be removed because they do not provide additional information and may give repeated observations unnecessary weight during model training.

### 5.2 Target Conflicts
Rows with identical feature combinations but different target values will be retained. There is not enough evidence to conclude that these observations are data errors.

### 5.3 Binary Features
All binary features contain valid `0` and `1` values. No changes are required.

### 5.4 Categorical and Ordinal Features
`Age`, `Education`, `Income`, and `GenHlth` contain values within their expected ranges. No changes are required.

### 5.5 BMI Outliers
BMI statistical outliers will be retained. Being an IQR-based outlier does not necessarily indicate an invalid or erroneous value.

### 5.6 MentHlth and PhysHlth Outliers
The identified statistical outliers will be retained because the values remain within the valid `0–30` day range of these features.

### 5.7 Missing Values
No missing values were detected, so no missing-value treatment is required.

### Final Cleaning Decision

The only confirmed cleaning action at this stage is the removal of exact duplicate rows. All other investigated observations will be retained because there is insufficient evidence that they represent data quality errors.

In [23]:
# Remove exact duplicate rows
df = df.drop_duplicates()

# Verify
print("Shape after removing duplicates:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())

Shape after removing duplicates: (229474, 22)
Remaining duplicate rows: 0


In [24]:
# Final Quality Check

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Target distribution:")
print(df["Diabetes_binary"].value_counts())

Dataset shape: (229474, 22)
Missing values: 0
Exact duplicates: 0
Target distribution:
Diabetes_binary
0.0    194377
1.0     35097
Name: count, dtype: int64


## 6. Cleaning Summary

The data quality investigation identified several potential issues, including exact duplicate rows, target conflicts, statistical outliers, and missing values.

### Key Findings

- Exact duplicate rows were identified and removed.
- Feature combinations with conflicting target values were identified but retained because there was insufficient evidence to classify them as data errors.
- All binary features contained valid `0` and `1` values.
- Categorical and ordinal features contained values within their expected ranges.
- Statistical outliers were identified in `BMI`, `MentHlth`, and `PhysHlth`, but they were retained because no evidence indicated that they were invalid observations.
- No missing values were detected.

### Final Cleaning Actions

The only data modification performed was the removal of exact duplicate rows.

The resulting dataset contains **229,474 rows and 22 columns**, with no exact duplicate rows and no missing values.

In [25]:
df.to_csv(
    "../data/processed/diabetes_cleaned.csv",
    index=False
)